# Subreddit groups v10 — banned/private subreddits relabelled from their archived bios

v9 labelled the 3,146 subreddits that are now banned or private (`status == 'banned or removed (no metadata)'`) from the
sample's archived posts or their name, because Reddit no longer shows their pages. Arctic Shift's subreddit dump
(`/data/reddit/subreddits_2025-01.zst`, read-only, owned by another user) kept the bio, sidebar and NSFW flag most of them
had before that, so v10 relabels them from what they said about themselves.

Order of precedence for these subreddits (first rule that applies wins):
1. **manual** overrides already in v9 (`MANUAL` below, kept as decided);
2. **profile** subreddits (`u_…`) stay `profile`;
3. **Reddit test subreddits** (`test_subreddit`) stay `other`, as decided for v6;
4. **adult by rule**: the dump's `over18` flag is True, or ≥ 90% of the archived posts in it were NSFW;
5. **agent label from the bio** (`subreddit_llm_labels_dumpbio_v10.csv`, guide `subreddit_classification_guide_archived_bio.md`) —
   unless the agent answered `unclear`;
6. otherwise the v9 label.

Subreddits that still exist keep their v9 label. `group_v9` keeps the previous label.
Output (never overwritten): `subreddit_review_output/subreddit_groups_v10.parquet` + review xlsx.
The dump extract is saved once as `subreddit_review_output/subreddit_dump_2025_01_v10.parquet`.

In [ ]:
# Libraries.
import pandas as pd
import numpy as np
import os
import re
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
import json
import io
import zstandard

# Folders (now the 998-account sample), the January 2025 subreddit dump, the group list, and the manual overrides.
SAMPLE_DIR = 'sample_998_v4_excl_output'
OUT_DIR = 'subreddit_review_output'
DUMP = '/data/reddit/subreddits_2025-01.zst'
GROUPS = ['profile', 'adult', 'crypto', 'politics', 'karma_farming', 'general', 'news', 'sales', 'location', 'entertainment', 'other']
NO_META = 'banned or removed (no metadata)'
MANUAL = {                                   # subreddit: (group, date decided) — carried over from v9
    'kidsvideos': ('entertainment', '2026-10-07'),
    'Twitch': ('entertainment', '2026-10-07'),
    'test23456': ('adult', '2026-10-07'),
}

# Write a file only if it does not exist yet (never overwrite); .xlsx goes to Excel, anything else to parquet.
def save_new(df, name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(f'exists, not overwritten: {path}')
        return
    if name.endswith('.xlsx'):
        df.to_excel(path, index=False, freeze_panes=(1, 1))
    else:
        df.to_parquet(path, engine='pyarrow')
    print(f'wrote {path}  {df.shape}')

# Collapse whitespace; None becomes an empty string.
clean = lambda x: re.sub(r'\s+', ' ', str(x or '')).strip()

## Dump extract
One pass over the 2.2 GB dump, keeping the lines for subreddits in the v9 list (matched case-insensitively). Skipped if the extract already exists.

In [ ]:
# Start from v9 (lowercase names for matching).
g = pd.read_parquet(os.path.join(OUT_DIR, 'subreddit_groups_v9.parquet'))
assert g['group'].isin(GROUPS).all()
g['sub_l'] = g['subreddit'].str.lower()

# Extract the dump rows for v9 subreddits once and cache them as parquet.
dump_path = os.path.join(OUT_DIR, 'subreddit_dump_2025_01_v10.parquet')
if not os.path.exists(dump_path):
    want, rows = set(g['sub_l']), []
    with open(DUMP, 'rb') as fh:
        reader = io.TextIOWrapper(zstandard.ZstdDecompressor(max_window_size=2**31).stream_reader(fh), encoding='utf-8')
        for line in reader:
            # Cheap string test on the name before parsing the JSON line.
            i = line.find('"display_name":"')
            if i < 0:
                continue
            if line[i + 16:line.find('"', i + 16)].lower() not in want:
                continue
            d = json.loads(line)
            rows.append({'sub_l': d['display_name'].lower(), 'd_name': d.get('display_name'), 'd_over18': d.get('over18'),
                         'd_type': d.get('subreddit_type'), 'd_title': d.get('title'), 'd_bio': d.get('public_description'),
                         'd_sidebar': d.get('description'), 'd_retrieved': d.get('retrieved_on'), 'd_created': d.get('created_utc')})
    save_new(pd.DataFrame(rows).drop_duplicates('sub_l', keep='last'), os.path.basename(dump_path))
dump = pd.read_parquet(dump_path)
print('v9 subreddits', len(g), '| found in dump', g['sub_l'].isin(dump['sub_l']).sum())

In [ ]:
# Join the dump rows onto v9.
m = g.merge(dump, on='sub_l', how='left')
assert len(m) == len(g)
# For subreddits PRAW returned nothing for: is there an archived bio or sidebar, and is it adult by rule
# (dump over18 flag, or at least 90% NSFW posts in the archived sample)?
no_meta = m['status'].eq(NO_META)
m['has_dump_bio'] = no_meta & (m['d_bio'].map(clean).ne('') | m['d_sidebar'].map(clean).ne(''))
m['rule_adult_flag'] = no_meta & m['d_over18'].eq(True)
m['rule_adult_nsfw'] = no_meta & m['archived_nsfw_share'].ge(0.9)
profile = m['subreddit'].str.startswith('u_')
print('no metadata', no_meta.sum(), '| found in dump', (no_meta & m['d_name'].notna()).sum(), '| with a bio', m['has_dump_bio'].sum())
print('adult by rule', (m['rule_adult_flag'] | m['rule_adult_nsfw']).sum(), '| flag only', (m['rule_adult_flag'] & ~m['rule_adult_nsfw']).sum())

# the agents saw exactly the no-metadata subreddits with a bio that are not adult by rule and not profiles
sent = m['has_dump_bio'] & ~(m['rule_adult_flag'] | m['rule_adult_nsfw']) & ~profile
lab = pd.read_csv(os.path.join(OUT_DIR, 'subreddit_llm_labels_dumpbio_v10.csv'), keep_default_na=False)
assert lab['subreddit'].is_unique and set(lab['subreddit']) == set(m.loc[sent, 'subreddit'])
assert lab['group'].isin(GROUPS + ['unclear']).all()
lab['group'].value_counts()

## Merge

In [ ]:
# Attach the agent labels made from the archived bios and keep the v9 group alongside.
lab = lab.rename(columns={'group': 'dump_agent_group', 'confidence': 'dump_confidence', 'reason': 'dump_reason'}).drop(columns='batch')
m = m.merge(lab, on='subreddit', how='left')
m['group_v9'] = m['group']
m['group_source_v9'] = m['group_source']

# Which step decides each subreddit: v9 by default; for no-metadata subreddits that are not manual,
# profile or test, the agent label, and then the adult rule (applied last, so it wins).
manual = m['subreddit'].isin(MANUAL)
test = m['test_subreddit'].fillna(False).astype(bool)
step = pd.Series('v9', index=m.index)
rule_adult = m['rule_adult_flag'] | m['rule_adult_nsfw']
agent = m['dump_agent_group'].notna() & m['dump_agent_group'].ne('unclear')
step[no_meta & ~manual & ~profile & ~test & agent] = 'agent'
step[no_meta & ~manual & ~profile & ~test & rule_adult] = 'rule_adult'

i = step.eq('agent')
m.loc[i, 'group'] = m.loc[i, 'dump_agent_group']
m.loc[i, 'group_source'] = 'agent_dump_bio'
i = step.eq('rule_adult')
m.loc[i, 'group'] = 'adult'
m.loc[i, 'group_source'] = np.where(m.loc[i, 'rule_adult_nsfw'], 'rule_archived_nsfw', 'rule_dump_over18')
# Manual overrides must survive unchanged.
for sr, (grp, when) in MANUAL.items():
    assert m.loc[m['subreddit'].eq(sr), 'group'].eq(grp).all(), sr
assert m['group'].isin(GROUPS).all()

# How many subreddits (and how many actions) changed group, and v9-to-v10 moves.
changed = m['group'].ne(m['group_v9'])
print('subreddits changed', changed.sum(), '| their actions', m.loc[changed, 'actions'].sum(), 'of', m['actions'].sum())
print('not changed because a rule came first: test subreddits with an agent label', (no_meta & test & agent).sum(),
      '| profiles adult by rule', (profile & rule_adult).sum(), '| test subreddits adult by rule', (test & rule_adult).sum())
pd.crosstab(m.loc[changed, 'group_v9'], m.loc[changed, 'group'], margins=True)

In [ ]:
# biggest moves, by the sample's pre-accusation actions in the subreddit
m.loc[changed].sort_values('actions', ascending=False)[
    ['subreddit', 'group_v9', 'group', 'group_source', 'dump_confidence', 'dump_reason', 'reason', 'actions', 'accounts']].head(40)

In [ ]:
# adult only because of the dump's over18 flag (the archived posts in them were not NSFW)
m.loc[step.eq('rule_adult') & ~m['rule_adult_nsfw'] & changed, ['subreddit', 'group_v9', 'archived_nsfw_share', 'd_bio', 'actions']]

## Group sizes in the sample (actions before the first accusation)

In [ ]:
# Every post and comment the sample made before its first (edit-adjusted) accusation, tagged with its subreddit's group.
accounts = pd.read_parquet(os.path.join(SAMPLE_DIR, 'sample_accounts.parquet')).set_index('username_key')
acts = []
for f in ['sample_posts.parquet', 'sample_comments.parquet']:
    # Drop repeated rows (repair files re-pulled some histories).
    d = pd.read_parquet(os.path.join(SAMPLE_DIR, f), columns=['outer_key', 'id', 'created_utc', 'subreddit']).drop_duplicates(['outer_key', 'id'])
    acts.append(d[d['created_utc'] < d['outer_key'].map(accounts['first_mention_adj_utc'])])
acts = pd.concat(acts)
summary = {}
# Group sizes under v9 and under v10, side by side.
for v in ['group_v9', 'group']:
    acts[v] = acts['subreddit'].map(m.set_index('subreddit')[v])
    summary[(v, 'subreddits')] = m[v].value_counts()
    summary[(v, 'actions')] = acts[v].value_counts()
    summary[(v, 'accounts')] = acts.groupby(v)['outer_key'].nunique()
summary = pd.DataFrame(summary).reindex(GROUPS)
print('actions with no group:', acts['group'].isna().sum())
summary

In [ ]:
# Save the groups, and a review sheet with a changed-since-v9 column.
out = m.drop(columns=['sub_l'])
save_new(out, 'subreddit_groups_v10.parquet')
review = out.sort_values(['group', 'actions'], ascending=[True, False])[
    ['subreddit', 'group', 'group_v9', 'group_source', 'group_source_v9', 'manual_override_date', 'dump_agent_group', 'dump_confidence',
     'dump_reason', 'confidence', 'reason', 'status', 'later_banned', 'private', 'test_subreddit', 'archived_nsfw_share', 'd_over18',
     'actions', 'accounts', 'd_title', 'd_bio', 'title', 'public_description']]
review.insert(2, 'changed', review['group'].ne(review['group_v9']))
save_new(review, 'subreddit_groups_v10_review.xlsx')